# Object Detection with PyTorch

This is a notebook from the [AI for Beginners Curriculum](http://aka.ms/ai-beginners). It complements [ObjectDetection.ipynb](ObjectDetection.ipynb), where we built a very simple box regressor with Keras.

In this notebook we will:

1. Implement **Intersection over Union (IoU)** from scratch and look at what it means.
2. Implement **Non-Maximum Suppression (NMS)** from scratch - the step that removes duplicate detections.
3. Compute **Average Precision (AP)** on a small synthetic example.
4. Run a **pretrained object detector from `torchvision`** on a real image, draw the boxes, and see the effect of the score threshold and of NMS.

Everything runs on a CPU in a few minutes. The only download is the pretrained model weights (13-75 MB, fetched automatically by `torchvision` on first use).

In [ ]:
import time

import numpy as np
import torch
import torchvision
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

np.random.seed(0)
torch.manual_seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"torch {torch.__version__}, torchvision {torchvision.__version__}, device: {device}")

## Boxes and IoU

A bounding box can be written in several ways. In the previous notebook we used `[x, y, w, h]`. `torchvision` detectors use **corner format** `[x1, y1, x2, y2]` (top-left and bottom-right corner, in pixels), so we will use it from now on.

**Intersection over Union** divides the area where two boxes overlap by the area covered by both of them together:

$$
IoU(A,B) = \frac{\mbox{area}(A \cap B)}{\mbox{area}(A \cup B)} = \frac{\mbox{area}(A \cap B)}{\mbox{area}(A)+\mbox{area}(B)-\mbox{area}(A \cap B)}
$$

Let's implement it with NumPy. The function works on a whole set of boxes at once, so that it returns a matrix with the IoU of every box in `a` against every box in `b`:

In [ ]:
def box_area(boxes):
    """Area of boxes given as an array of shape (N, 4) in [x1, y1, x2, y2] format."""
    return (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])


def iou_matrix(a, b):
    """IoU of every box in `a` (N, 4) with every box in `b` (M, 4). Returns an (N, M) matrix."""
    a = np.asarray(a, dtype=np.float32).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float32).reshape(-1, 4)
    # corners of the intersection rectangle (broadcasting gives an (N, M) grid)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    # if the boxes do not overlap, width or height is negative -> clip to 0
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    union = box_area(a)[:, None] + box_area(b)[None, :] - inter
    return inter / np.maximum(union, 1e-9)


def iou(box1, box2):
    """IoU of two single boxes."""
    return float(iou_matrix([box1], [box2])[0, 0])


print(iou([0, 0, 10, 10], [0, 0, 10, 10]))    # identical boxes -> 1.0
print(iou([0, 0, 10, 10], [20, 20, 30, 30]))  # disjoint boxes  -> 0.0
print(iou([0, 0, 10, 10], [5, 0, 15, 10]))    # half shifted: intersection 50, union 150 -> 0.333

✅ **Thought prompt**: why is IoU 1/3 and not 1/2 for the last pair of boxes? (Hint: the union counts the overlapping area only once.)

To get a feeling for the numbers, let's draw a "ground truth" box in green and a few predictions that are shifted more and more. We will reuse the small drawing helper below throughout the notebook.

In [ ]:
def draw_box(ax, box, color='r', label=None, lw=2, ls='-'):
    """Draw one [x1, y1, x2, y2] box on a matplotlib axis."""
    x1, y1, x2, y2 = box
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color, lw=lw, ls=ls))
    if label:
        ax.text(x1, y1 - 3, label, color='white', fontsize=9, va='bottom',
                bbox=dict(facecolor=color, alpha=0.8, pad=1.5, ec='none'))


gt = [20, 20, 120, 100]
shifts = [0, 10, 30, 60]

fig, axes = plt.subplots(1, len(shifts), figsize=(14, 3))
for ax, s in zip(axes, shifts):
    pred = [20 + s, 20 + s // 2, 120 + s, 100 + s // 2]
    draw_box(ax, gt, 'green', lw=3)
    draw_box(ax, pred, 'red')
    ax.set_xlim(0, 200); ax.set_ylim(150, 0); ax.set_aspect('equal')
    ax.set_title(f"IoU = {iou(gt, pred):.2f}")
plt.show()

In the PASCAL VOC benchmark a detection usually counts as correct when $IoU \geq 0.5$ (COCO averages over several thresholds from 0.5 to 0.95). Looking at the pictures above - would you be happy with a box that has IoU = 0.5?

## Non-Maximum Suppression (NMS)

Most detectors predict many candidate boxes, and several of them usually land on the same object, each with a slightly different position and score. If we showed all of them we would get a cluster of boxes around every object.

**Non-Maximum Suppression** cleans this up with a simple greedy procedure:

1. Sort all boxes by score, highest first.
2. Keep the best box.
3. Remove every remaining box whose IoU with the kept box is above a threshold (typically 0.5) - they most likely describe the same object.
4. Repeat from step 2 with the boxes that are left.

Here is a NumPy version:

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """Greedy non-maximum suppression. Returns the indices of the boxes to keep (best score first)."""
    boxes = np.asarray(boxes, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)
    order = np.argsort(-scores)            # indices sorted by score, best first
    keep = []
    while order.size > 0:
        best = order[0]
        keep.append(int(best))
        if order.size == 1:
            break
        overlaps = iou_matrix(boxes[best:best + 1], boxes[order[1:]])[0]
        order = order[1:][overlaps <= iou_threshold]   # drop boxes that overlap the best one too much
    return keep

Let's test it on made-up detections. We place two "objects" in the picture and simulate a detector that fires several jittered boxes around each of them, plus one low-confidence false alarm in the background:

In [ ]:
rng = np.random.default_rng(0)

true_objects = np.array([[40, 40, 140, 160],
                         [180, 70, 280, 150]], dtype=np.float32)

det_boxes, det_scores = [], []
for obj, base_score in zip(true_objects, [0.95, 0.85]):
    for k in range(5):
        det_boxes.append(obj + rng.normal(0, 6, size=4))     # jitter the box corners
        det_scores.append(base_score - 0.1 * k + rng.normal(0, 0.02))
det_boxes.append([210, 150, 260, 200]); det_scores.append(0.30)  # a false alarm
det_boxes = np.array(det_boxes, dtype=np.float32)
det_scores = np.array(det_scores, dtype=np.float32)

keep = nms(det_boxes, det_scores, iou_threshold=0.5)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, idx, title in [(axes[0], range(len(det_boxes)), f"Before NMS: {len(det_boxes)} boxes"),
                       (axes[1], keep, f"After NMS (IoU threshold 0.5): {len(keep)} boxes")]:
    for i in idx:
        draw_box(ax, det_boxes[i], 'tab:red', f"{det_scores[i]:.2f}", lw=1.5)
    ax.set_xlim(0, 320); ax.set_ylim(220, 0); ax.set_aspect('equal'); ax.set_title(title)
plt.show()

NMS keeps the best box of each cluster. The false alarm survives because it has a low IoU with the other boxes - NMS does not look at scores in absolute terms, that is the job of the *score threshold* (we will use one in a moment).

`torchvision` already ships a fast implementation, `torchvision.ops.nms`. Let's check that ours returns the same boxes, and see how the IoU threshold changes the result:

In [ ]:
tv_keep = torchvision.ops.nms(torch.tensor(det_boxes), torch.tensor(det_scores), iou_threshold=0.5)
print("ours:        ", keep)
print("torchvision: ", tv_keep.tolist())
assert keep == tv_keep.tolist()

for thr in [0.1, 0.3, 0.5, 0.7, 0.9]:
    print(f"IoU threshold {thr}: keeps {len(nms(det_boxes, det_scores, thr))} of {len(det_boxes)} boxes")

✅ **Thought prompt**: a *low* threshold removes boxes aggressively, a *high* one keeps almost everything. What could go wrong with a very low threshold in a picture with two people standing close to each other? (This is a known weakness of NMS in crowded scenes.)

## Average Precision (AP) from scratch

Now let's measure how good a detector is. We need ground-truth boxes and detections with scores. To keep it small we will invent them: three images, one class, a handful of boxes per image. Each ground-truth box gets one jittered detection (some jitter is big enough to make it a bad box), and every image gets a couple of random false detections.

In [ ]:
rng = np.random.default_rng(1)

gt_boxes = {}      # image id -> array of ground-truth boxes
detections = []    # (image id, box, score)
for img_id in range(3):
    n_obj = rng.integers(2, 4)
    xy = rng.uniform(0, 200, size=(n_obj, 2))
    wh = rng.uniform(50, 100, size=(n_obj, 2))
    gt_boxes[img_id] = np.hstack([xy, xy + wh]).astype(np.float32)
    for b in gt_boxes[img_id]:
        if rng.random() < 0.85:                                     # most objects are found...
            jitter = rng.normal(0, rng.choice([3, 12]), size=4)     # ...some of them imprecisely
            detections.append((img_id, b + jitter, rng.uniform(0.5, 1.0)))
    for _ in range(2):                                               # false alarms
        xy = rng.uniform(0, 250, size=2)
        detections.append((img_id, np.hstack([xy, xy + rng.uniform(40, 90, size=2)]), rng.uniform(0.1, 0.8)))

n_gt = sum(len(b) for b in gt_boxes.values())
print(f"{n_gt} ground-truth boxes, {len(detections)} detections")

To evaluate, we go through the detections from the most to the least confident one. A detection is a **true positive (TP)** if it overlaps a not-yet-matched ground-truth box of the same image with $IoU \geq$ threshold, otherwise it is a **false positive (FP)**. After each detection we compute

* **precision** = TP / (TP + FP) = how many of the detections so far are correct
* **recall** = TP / (number of ground-truth boxes) = how many of the real objects we have found so far

Plotting precision against recall gives the precision-recall curve from the lesson, and **AP** is the area under it.

In [ ]:
def precision_recall(detections, gt_boxes, iou_threshold=0.5):
    """Precision and recall after each detection, going from the most to the least confident one."""
    detections = sorted(detections, key=lambda d: -d[2])
    matched = {i: np.zeros(len(b), dtype=bool) for i, b in gt_boxes.items()}
    tp = np.zeros(len(detections))
    for k, (img_id, box, score) in enumerate(detections):
        overlaps = iou_matrix([box], gt_boxes[img_id])[0]
        j = overlaps.argmax()
        if overlaps[j] >= iou_threshold and not matched[img_id][j]:
            matched[img_id][j] = True      # every ground-truth box can be found only once
            tp[k] = 1
    tp_cum = np.cumsum(tp)
    precision = tp_cum / np.arange(1, len(detections) + 1)
    recall = tp_cum / sum(len(b) for b in gt_boxes.values())
    return precision, recall


def average_precision(precision, recall, points=11):
    """Area under the precision-recall curve. points=11: the PASCAL VOC 2007 style used in the lesson;
    points=None: exact area under the curve with the precision made monotonically decreasing."""
    if points:
        # average of the best precision reachable at recall >= 0, 0.1, ..., 1.0
        return np.mean([precision[recall >= t].max() if (recall >= t).any() else 0
                        for t in np.linspace(0, 1, points)])
    p = np.concatenate([[0], precision, [0]])
    r = np.concatenate([[0], recall, [1]])
    p = np.maximum.accumulate(p[::-1])[::-1]          # precision envelope
    return float(np.sum((r[1:] - r[:-1]) * p[1:]))


fig, ax = plt.subplots(figsize=(5, 4))
for thr in [0.5, 0.75]:
    p, r = precision_recall(detections, gt_boxes, thr)
    ap = average_precision(p, r)
    print(f"IoU threshold {thr}: AP (11 points) = {ap:.3f}, AP (exact area) = {average_precision(p, r, None):.3f}")
    ax.plot(r, p, marker='o', ms=3, label=f"IoU >= {thr}  (AP={ap:.2f})")
ax.set_xlabel('recall'); ax.set_ylabel('precision'); ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)
ax.legend(); ax.grid(alpha=0.3); plt.show()

The stricter the IoU threshold, the fewer detections count as correct, and the lower the AP gets - this is why COCO reports AP averaged over several IoU thresholds. **mAP** is simply this AP averaged over all object classes.

For real benchmarks you would not write this yourself but use a tested implementation such as [pycocotools](https://github.com/cocodataset/cocoapi) or [torchmetrics](https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html).

## Running a pretrained detector

Training a detector from scratch needs a large annotated dataset and a lot of compute. Fortunately `torchvision` provides detectors that were already trained on [COCO](http://cocodataset.org/#home) (80 object classes). The weights are downloaded automatically the first time and cached.

We will start with **Faster R-CNN with a MobileNetV3 backbone** - a two-stage detector (region proposals + classification) that is small enough for a laptop CPU. The helper below loads any detector from `torchvision.models.detection` together with its class names:

In [ ]:
from torchvision.models import get_model, get_model_weights


def load_detector(name):
    """Load a torchvision detector with COCO-pretrained weights. Returns (model, class names)."""
    weights = get_model_weights(name).DEFAULT
    model = get_model(name, weights=weights).to(device).eval()
    return model, weights.meta["categories"]


model, categories = load_detector("fasterrcnn_mobilenet_v3_large_fpn")
print(f"{len(categories)} entries in the class list (COCO has 80 classes, the unused slots are called 'N/A')")
print("first classes:", categories[1:6])

Let's load the sample image from the previous notebook. Detection models in `torchvision` expect a float tensor of shape `(3, H, W)` with values in $[0,1]$ and take care of resizing and normalization themselves:

In [ ]:
IMAGE_PATH = 'images/1200px-Girl_and_cat.jpg'   # try your own picture here!

pil_img = Image.open(IMAGE_PATH).convert('RGB')
img_tensor = torchvision.transforms.functional.to_tensor(pil_img)   # (3, H, W), values in [0, 1]
print(img_tensor.shape)

plt.imshow(pil_img); plt.axis('off'); plt.show()

The model takes a *list* of images and returns, for each image, a dictionary with the `boxes` (in `[x1, y1, x2, y2]` format), the predicted `labels` (indices into the COCO class list) and the confidence `scores`:

In [ ]:
@torch.no_grad()
def detect(model, img_tensor):
    """Run a detector on one image tensor and return its predictions as a dict of CPU tensors."""
    out = model([img_tensor.to(device)])[0]
    return {k: v.cpu() for k, v in out.items()}


pred = detect(model, img_tensor)
print(f"{len(pred['boxes'])} detections with a score above the model's built-in minimum")
for box, label, score in list(zip(pred['boxes'], pred['labels'], pred['scores']))[:8]:
    print(f"{categories[label]:>10s}  score={score:.2f}  box={[round(v) for v in box.tolist()]}")

Now let's draw the detections. Models return many low-confidence boxes, so we only show those above a **score threshold**:

In [ ]:
def show_detections(ax, pil_img, pred, categories, score_threshold=0.5, title=None):
    """Draw all detections with score >= score_threshold on top of the image."""
    ax.imshow(pil_img); ax.axis('off')
    colors = plt.cm.tab10.colors
    shown = 0
    for box, label, score in zip(pred['boxes'], pred['labels'], pred['scores']):
        if score < score_threshold:
            continue
        draw_box(ax, box.tolist(), colors[int(label) % 10], f"{categories[label]} {score:.2f}")
        shown += 1
    ax.set_title(title or f"score >= {score_threshold}: {shown} boxes")


fig, ax = plt.subplots(figsize=(8, 8))
show_detections(ax, pil_img, pred, categories, score_threshold=0.5)
plt.show()

### Effect of the score threshold

A low threshold finds more objects (higher recall) but also more false alarms (lower precision); a high threshold does the opposite. This is exactly the trade-off that the precision-recall curve describes. Let's compare three thresholds:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, thr in zip(axes, [0.1, 0.5, 0.9]):
    show_detections(ax, pil_img, pred, categories, score_threshold=thr)
plt.tight_layout(); plt.show()

✅ **Thought prompt**: which threshold would you choose for counting people entering a shop, and which one for a safety system that must not miss anything? Does your answer change if a false alarm is cheap or expensive?

### NMS inside the detector

The model you just ran already applies the NMS you wrote above as its last step: `model.roi_heads.nms_thresh` holds the IoU threshold (0.5 by default). Let's switch NMS off (an IoU threshold of 1.0 never suppresses anything) and see how many raw boxes the network really produces:

In [ ]:
default_nms = model.roi_heads.nms_thresh
model.roi_heads.nms_thresh = 1.0          # IoU can never exceed 1.0, so nothing is suppressed
pred_no_nms = detect(model, img_tensor)
model.roi_heads.nms_thresh = default_nms  # restore the default

print(f"with NMS ({default_nms}): {int((pred['scores'] >= 0.8).sum())} boxes above score 0.8")
print(f"without NMS:       {int((pred_no_nms['scores'] >= 0.8).sum())} boxes above score 0.8")

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
show_detections(axes[0], pil_img, pred, categories, 0.8, title="with NMS")
show_detections(axes[1], pil_img, pred_no_nms, categories, 0.8, title="without NMS: duplicates appear")
plt.tight_layout(); plt.show()

### Speed vs. accuracy

Detectors differ a lot in size and speed. Here we compare the model above with `ssdlite320_mobilenet_v3_large`, a **one-pass** (SSD-style) detector that is much smaller and faster, but also less accurate. Other options you can try in the list are `retinanet_resnet50_fpn` and `fasterrcnn_resnet50_fpn` (larger downloads, better accuracy, slower on CPU).

In [ ]:
model_names = ["ssdlite320_mobilenet_v3_large", "fasterrcnn_mobilenet_v3_large_fpn"]

fig, axes = plt.subplots(1, len(model_names), figsize=(8 * len(model_names), 8))
for ax, name in zip(axes, model_names):
    m, cats = load_detector(name)
    detect(m, img_tensor)                         # warm-up run
    t0 = time.time()
    p = detect(m, img_tensor)
    dt = time.time() - t0
    n_params = sum(x.numel() for x in m.parameters()) / 1e6
    show_detections(ax, pil_img, p, cats, 0.5, title=f"{name}\n{n_params:.1f}M parameters, {dt:.2f} s per image")
plt.tight_layout(); plt.show()

## What's next?

* **YOLO family and other one-pass detectors.** Models such as the YOLO series (e.g. [Ultralytics YOLO](https://docs.ultralytics.com/)), SSD and RetinaNet predict boxes and classes in a single pass and are the usual choice for real-time applications.
* **Transformer detectors.** [DETR](https://arxiv.org/abs/2005.12872) treats detection as predicting a *set* of boxes with a transformer and needs no NMS at all. You can try it with the Hugging Face [`transformers`](https://huggingface.co/docs/transformers/tasks/object_detection) library.
* **Your own classes.** To detect objects that are not in COCO you *fine-tune* a pretrained detector on your own annotated images - see the [TorchVision Object Detection Finetuning tutorial](https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html) and the [lab assignment](lab/README.md) of this lesson.

**Challenge**: change `IMAGE_PATH` to a photo of your own (or download one with `urllib.request.urlretrieve`) and run the detector on it. Which objects does it miss or confuse? Then draw one box by hand around an object, and use `iou()` to compare it with the box predicted by the model.